# 18 — Sadhwani et al. (2021): Deep Learning for Mortgage Risk

Neural network model for monthly mortgage state transitions.

**Reference**: Giesecke, Sirignano & Sadhwani (2021). "Deep Learning for Mortgage Risk." *Journal of Financial Econometrics*, 19(2), 313–368.

**Model**: Feedforward neural network with softmax output predicting monthly transition probabilities P[state_t | X_{t-1}] over 3 states (Current, Prepay, Default). Ensemble of 8 independently trained networks with bootstrapped data.

**Architecture**: 5 hidden layers (200-140-140-140-140), ReLU activation, dropout, L2 regularization.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# Project imports
sys.path.insert(0, str(Path('.').resolve().parent / 'src'))
from competing_risks.sadhwani_net import (
    SadhwaniNet, SadhwaniEnsemble, get_device,
    train_single_model, compute_cif, compute_cif_frozen, compute_cif_ar,
    fit_ar_models, variable_sensitivity,
    prepare_monthly_targets, prepare_features,
    ALL_FEATURES, TRAIN_FOLDS, VAL_FOLDS, TEST_FOLD,
)
from competing_risks.evaluation import (
    time_dependent_concordance_index,
    brier_score_competing_risks,
    EVAL_TIMES,
)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print(f'PyTorch {torch.__version__}')
device = get_device()
print(f'Device: {device}')

## 1. Configuration

In [ ]:
# Hyperparameters (paper's cross-validated optimum)
HIDDEN_SIZES = [200, 140, 140, 140, 140]  # 5 hidden layers
DROPOUT = 0.5
LR = 0.1
LR_HALFLIFE = 800   # Eq. 9: lr_t = lr_0 / (1 + t/800)
WEIGHT_DECAY = 1e-4  # L2 penalty
BATCH_SIZE = 4096
N_EPOCHS = 100
PATIENCE = 10
N_ENSEMBLE = 8
SEED = 42

DATA_DIR = Path('..') / 'data' / 'processed'
MODELS_DIR = Path('..') / 'models'

## 2. Data Loading

In [ ]:
panel_df = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')
print(f'Loaded {len(panel_df):,} loan-months, {panel_df["loan_sequence_number"].nunique():,} loans')

# Prepare features (log_upb, bal_repaid_lag1)
feature_cols, panel_df = prepare_features(panel_df)
print(f'Features ({len(feature_cols)}): {feature_cols}')

# Monthly transition targets
panel_df = prepare_monthly_targets(panel_df)
panel_df = panel_df.dropna(subset=feature_cols)

print(f'\nTarget distribution:')
print(panel_df['target'].value_counts().sort_index().rename({0: 'Current', 1: 'Prepay', 2: 'Default'}))

## 3. Train / Val / Test Split

In [ ]:
train_panel = panel_df[panel_df['fold'].isin(TRAIN_FOLDS)].copy()
val_panel = panel_df[panel_df['fold'].isin(VAL_FOLDS)].copy()
test_panel = panel_df[panel_df['fold'] == TEST_FOLD].copy()

for name, split in [('Train', train_panel), ('Val', val_panel), ('Test', test_panel)]:
    n_loans = split['loan_sequence_number'].nunique()
    terminal = split.groupby('loan_sequence_number').last()
    print(f'{name}: {len(split):,} loan-months ({n_loans:,} loans) — '
          f'censored={int((terminal["event_code"]==0).sum()):,}, '
          f'prepay={int((terminal["event_code"]==1).sum()):,}, '
          f'default={int((terminal["event_code"]==2).sum()):,}')

## 4. Standardize Features

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_panel[feature_cols].values).astype(np.float32)
y_train = train_panel['target'].values.astype(np.int64)

X_val = scaler.transform(val_panel[feature_cols].values).astype(np.float32)
y_val = val_panel['target'].values.astype(np.int64)

X_test = scaler.transform(test_panel[feature_cols].values).astype(np.float32)
y_test = test_panel['target'].values.astype(np.int64)

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'Class balance (train): current={np.mean(y_train==0):.4f}, '
      f'prepay={np.mean(y_train==1):.4f}, default={np.mean(y_train==2):.4f}')

## 5. Train Single 5-Layer Network

First train a single model to inspect loss curves and convergence.

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

single_net = SadhwaniNet(
    n_features=len(feature_cols),
    hidden_sizes=HIDDEN_SIZES,
    n_states=3,
    dropout=DROPOUT,
)
print(single_net)
n_params = sum(p.numel() for p in single_net.parameters())
print(f'Parameters: {n_params:,}')

train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                         torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                       torch.tensor(y_val, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

history = train_single_model(
    single_net, train_loader, val_loader,
    lr=LR, weight_decay=WEIGHT_DECAY,
    n_epochs=N_EPOCHS, lr_halflife=LR_HALFLIFE,
    patience=PATIENCE, device=device,
)

In [ ]:
# Loss curves
fig, ax = plt.subplots()
ax.plot(history['train_loss'], label='Train')
ax.plot(history['val_loss'], label='Validation')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Training curves — single 5-layer network')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best val loss: {min(history["val_loss"]):.6f} '
      f'(epoch {history["val_loss"].index(min(history["val_loss"]))+1})')

## 6. Depth Comparison (Table 11)

Compare networks of different depth (0, 1, 3, 5 hidden layers) to assess the value of nonlinearity.

In [ ]:
depth_configs = {
    '0-Hidden (logit)': [],
    '1-Hidden': [200],
    '3-Hidden': [200, 140, 140],
    '5-Hidden': [200, 140, 140, 140, 140],
}

depth_results = []
for name, hsizes in depth_configs.items():
    print(f'\nTraining {name}...')
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    net = SadhwaniNet(n_features=len(feature_cols), hidden_sizes=hsizes,
                      n_states=3, dropout=DROPOUT if hsizes else 0.0)
    ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                       torch.tensor(y_train, dtype=torch.long))
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    hist = train_single_model(
        net, loader, val_loader,
        lr=LR, weight_decay=WEIGHT_DECAY, n_epochs=N_EPOCHS,
        lr_halflife=LR_HALFLIFE, patience=PATIENCE, device=device, verbose=False,
    )
    depth_results.append({
        'Model': name,
        'Best train loss': min(hist['train_loss']),
        'Best val loss': min(hist['val_loss']),
        'Epochs': len(hist['train_loss']),
    })

depth_df = pd.DataFrame(depth_results)
depth_df

## 7. Ensemble Training

Train 8 independently initialised 5-layer networks, each on bootstrapped training data. The paper shows diminishing returns beyond 8 members.

In [ ]:
ensemble = SadhwaniEnsemble(
    n_models=N_ENSEMBLE,
    n_features=len(feature_cols),
    hidden_sizes=HIDDEN_SIZES,
    n_states=3,
    dropout=DROPOUT,
)

histories = ensemble.fit(
    X_train, y_train, X_val, y_val,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    lr_halflife=LR_HALFLIFE,
    patience=PATIENCE,
    device=device,
    base_seed=SEED,
)

In [ ]:
# Per-member validation loss
for i, hist in enumerate(histories):
    best = min(hist['val_loss'])
    ep = hist['val_loss'].index(best) + 1
    print(f'Member {i+1}: best val_loss = {best:.6f} (epoch {ep})')

# Ensemble cross-entropy
val_probs = ensemble.predict_proba(X_val, device=device)
val_ce = -np.mean(np.log(val_probs[np.arange(len(y_val)), y_val].clip(1e-10)))
print(f'\nEnsemble val cross-entropy: {val_ce:.6f}')

In [ ]:
# Save ensemble
ensemble.save(str(MODELS_DIR / 'sadhwani_ensemble.pt'))
print('Ensemble saved.')

# Save scaler
import pickle
with open(MODELS_DIR / 'sadhwani_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
ensemble = SadhwaniEnsemble.load(str(MODELS_DIR / 'sadhwani_ensemble.pt'), device=device)
print(f'Loaded ensemble: {ensemble.n_models} models, {ensemble.n_features} features')

## 8. Monthly Transition Predictions on Test Set

In [ ]:
# Predict on test loan-months
test_probs = ensemble.predict_proba(X_test, device=device)
print(f'Test predictions shape: {test_probs.shape}')
print(f'Mean predicted probabilities:')
print(f'  P(Current) = {test_probs[:, 0].mean():.6f}')
print(f'  P(Prepay)  = {test_probs[:, 1].mean():.6f}')
print(f'  P(Default) = {test_probs[:, 2].mean():.6f}')

# Cross-entropy on test
test_ce = -np.mean(np.log(test_probs[np.arange(len(y_test)), y_test].clip(1e-10)))
print(f'\nTest cross-entropy: {test_ce:.6f}')

## 9. Cumulative Incidence Functions

The model predicts one-month-ahead transition probabilities. To evaluate at horizons
$\tau = 24, 48, 72$ months we chain monthly predictions into CIF:

$$\text{CIF}_k(t) = \text{CIF}_k(t-1) + S(t-1) \cdot p_k(t), \qquad S(t) = S(t-1) \cdot p_0(t)$$

**Key issue**: chaining requires features for every future month. We implement three approaches:

| Method | Macro features | Behavioural features | Future info? |
|--------|---------------|---------------------|-------------|
| **Observed** (`compute_cif`) | From panel | From panel | Yes (diagnostic only) |
| **Frozen** (`compute_cif_frozen`) | Held at time-zero | Mechanical update | No |
| **AR-simulated** (`compute_cif_ar`) | AR(p) forward paths | Mechanical update | No |

In [ ]:
# --- Method A: Observed features (uses future info — diagnostic only) ---
print('Method A: Observed features (diagnostic)...')
cif_observed = compute_cif(
    test_panel, feature_cols, ensemble, scaler,
    eval_times=EVAL_TIMES, device=device, batch_size=BATCH_SIZE,
)

# --- Method B: Frozen features (no future info) ---
print('Method B: Frozen features...')
cif_frozen = compute_cif_frozen(
    test_panel, feature_cols, ensemble, scaler,
    eval_times=EVAL_TIMES, device=device, batch_size=BATCH_SIZE,
)

# --- Method C: AR-simulated macro paths (no future info) ---
print('Method C: Fitting AR models on training panel...')
ar_models = fit_ar_models(train_panel, feature_cols, max_lag=4)
print(f'  AR models fitted for: {list(ar_models.keys())}')
for col, m in ar_models.items():
    p = len(m.params) - 1
    print(f'    {col}: AR({p}), sigma={np.sqrt(m.sigma2):.4f}')

print('Method C: Simulating CIF with AR paths (50 simulations)...')
cif_ar = compute_cif_ar(
    test_panel, feature_cols, ensemble, scaler,
    eval_times=EVAL_TIMES, ar_models=ar_models,
    n_simulations=50, device=device, batch_size=BATCH_SIZE, seed=SEED,
)

# Summary
for name, cif in [('Observed', cif_observed), ('Frozen', cif_frozen), ('AR-simulated', cif_ar)]:
    print(f'\n{name}:')
    for t in EVAL_TIMES:
        print(f'  t={t}: mean CIF_prepay={cif[f"cif_prepay_{t}"].mean():.4f}, '
              f'mean CIF_default={cif[f"cif_default_{t}"].mean():.4f}')

### Per-loan AR simulation — single loan example

Visualise how the AR model extends each loan's observed macro history into the future.
Each loan starts its AR simulation from its own last observed values, preserving
cross-loan variation in macro conditions.

In [ ]:
from competing_risks.sadhwani_net import simulate_ar_paths_per_loan, MACRO_FEATURES

# Pick a test loan with long history for a meaningful visualisation
test_sorted = test_panel.sort_values(['loan_sequence_number', 'loan_age'])
loan_lengths = test_sorted.groupby('loan_sequence_number').size()
long_loans = loan_lengths[loan_lengths >= 30].index

# Select the loan with the widest HPI range
best_loan, best_range = None, 0
for lid in long_loans[:200]:
    vals = test_sorted.loc[test_sorted['loan_sequence_number'] == lid, 'hpi_st_d_t_o']
    r = vals.max() - vals.min()
    if r > best_range:
        best_range, best_loan = r, lid

loan_data = test_sorted[test_sorted['loan_sequence_number'] == best_loan]
last_age = int(loan_data['loan_age'].iloc[-1])
print(f'Loan {best_loan}: {len(loan_data)} months observed (age 1–{last_age})')

# Build per-loan AR start values from the last p observations
start_values = {}
for col in MACRO_FEATURES:
    if col not in ar_models:
        continue
    p = len(ar_models[col].params) - 1
    hist_vals = loan_data[col].values
    init = hist_vals[-p:] if len(hist_vals) >= p else np.full(p, hist_vals[-1])
    start_values[col] = init.reshape(1, p).astype(np.float64)

# Simulate 50 paths forward for 48 months
n_forward = 48
ar_paths = simulate_ar_paths_per_loan(
    {k: v for k, v in ar_models.items() if k in MACRO_FEATURES},
    start_values, n_steps=n_forward, n_simulations=50, seed=SEED,
)

# Plot: observed history + AR-simulated future for each macro feature
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, col in zip(axes.flat, MACRO_FEATURES):
    hist_ages = loan_data['loan_age'].values.astype(int)
    hist_vals = loan_data[col].values
    ax.plot(hist_ages, hist_vals, 'b-', linewidth=2, label='Observed (this loan)')
    ax.plot(hist_ages[-1], hist_vals[-1], 'bo', markersize=8)

    if col in ar_paths:
        sims = ar_paths[col][:, 0, :]  # (50, n_forward)
        future_ages = np.arange(last_age + 1, last_age + 1 + n_forward)
        for s in range(50):
            ax.plot(future_ages, sims[s], 'r-', alpha=0.1, linewidth=0.5)
        ax.plot(future_ages, sims.mean(axis=0), 'r-', linewidth=2, label='AR mean (50 sims)')
        ax.fill_between(future_ages, np.percentile(sims, 5, axis=0),
                        np.percentile(sims, 95, axis=0), color='red', alpha=0.15,
                        label='90% CI')

    ax.axvline(last_age, color='gray', linestyle='--', alpha=0.5, label='Prediction start')
    p_order = len(ar_models[col].params) - 1
    sigma = np.sqrt(ar_models[col].sigma2)
    ax.set_title(f'{col} — AR({p_order}), σ={sigma:.2f}')
    ax.set_xlabel('Loan age (months)')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Per-loan AR simulation — loan {best_loan} '
             f'(observed {last_age} months, forecasting {n_forward})', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Time-Dependent Concordance Index — All Three Methods

In [ ]:
def eval_cindex(cif_result, label):
    """Evaluate C-index for a CIF result dict."""
    rows = []
    for cause, code in [('Prepay', 1), ('Default', 2)]:
        row = {'Method': label, 'Cause': cause}
        for t in EVAL_TIMES:
            c_idx, _, n_comp = time_dependent_concordance_index(
                cif_result['duration'], cif_result['event_code'],
                cif_result[f'cif_{cause.lower()}_{t}'],
                t, event_of_interest=code,
            )
            row[f'C({t})'] = c_idx
        rows.append(row)
    return rows

all_cindex = []
for label, cif in [('Observed', cif_observed), ('Frozen', cif_frozen), ('AR-simulated', cif_ar)]:
    all_cindex.extend(eval_cindex(cif, label))

cindex_df = pd.DataFrame(all_cindex)
print('Time-dependent C-index comparison:')
for cause in ['Prepay', 'Default']:
    print(f'\n  {cause}:')
    sub = cindex_df[cindex_df['Cause'] == cause].set_index('Method')
    print(sub[[f'C({t})' for t in EVAL_TIMES]].to_string())

cindex_df

## 11. Brier Score — All Three Methods

In [ ]:
def eval_brier(cif_result, label):
    """Evaluate Brier score for a CIF result dict."""
    rows = []
    for cause, code in [('Prepay', 1), ('Default', 2)]:
        row = {'Method': label, 'Cause': cause}
        for t in EVAL_TIMES:
            bs = brier_score_competing_risks(
                cif_result['duration'], cif_result['event_code'],
                cif_result[f'cif_{cause.lower()}_{t}'],
                t, event_of_interest=code,
            )
            row[f'BS({t})'] = bs
        rows.append(row)
    return rows

all_brier = []
for label, cif in [('Observed', cif_observed), ('Frozen', cif_frozen), ('AR-simulated', cif_ar)]:
    all_brier.extend(eval_brier(cif, label))

brier_df = pd.DataFrame(all_brier)
print('Brier Score comparison:')
for cause in ['Prepay', 'Default']:
    print(f'\n  {cause}:')
    sub = brier_df[brier_df['Cause'] == cause].set_index('Method')
    print(sub[[f'BS({t})' for t in EVAL_TIMES]].to_string())

brier_df

## 12. Variable Sensitivity Analysis

Following Eq. (7) of the paper: average absolute gradient of transition probability w.r.t. each feature (finite-difference approximation).

In [ ]:
# Use val set subsample for speed
n_sens = min(50000, len(X_val))
idx_sens = np.random.RandomState(SEED).choice(len(X_val), n_sens, replace=False)
X_sens = X_val[idx_sens]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (cause, to_state) in zip(axes, [('Prepay', 1), ('Default', 2)]):
    sens = variable_sensitivity(ensemble, X_sens, feature_cols,
                                to_state=to_state, device=device)
    ax.barh(sens['feature'], sens['sensitivity'])
    ax.set_xlabel('Average |gradient|')
    ax.set_title(f'Sensitivity: Current → {cause}')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 13. Partial Dependence Plots

Show the nonlinear relationship between key features and transition probabilities, holding all other covariates at their mean value.

In [ ]:
# Partial dependence: vary one feature, fix others at mean
n_grid = 100
top_features = ['int_rate', 'log_upb', 'fico_score', 'hpi_st_d_t_o',
                't_del_30d_12m', 'bal_repaid_lag1']
top_features = [f for f in top_features if f in feature_cols]

fig, axes = plt.subplots(2, len(top_features), figsize=(5*len(top_features), 8),
                         squeeze=False)

# Base point: mean of all features (= 0 in standardised space)
X_base = np.zeros((n_grid, len(feature_cols)), dtype=np.float32)

for j, feat in enumerate(top_features):
    feat_idx = feature_cols.index(feat)
    # Range: -3 to +3 std in standardised space
    grid = np.linspace(-3, 3, n_grid)
    X_pd = X_base.copy()
    X_pd[:, feat_idx] = grid

    probs = ensemble.predict_proba(X_pd, device=device)

    # Convert grid back to original scale for x-axis
    x_orig = grid * scaler.scale_[feat_idx] + scaler.mean_[feat_idx]

    for row, (cause, state_idx) in enumerate([("Prepay", 1), ("Default", 2)]):
        axes[row, j].plot(x_orig, probs[:, state_idx], 'b-', linewidth=2)
        axes[row, j].set_xlabel(feat)
        if j == 0:
            axes[row, j].set_ylabel(f'P({cause})')
        axes[row, j].set_title(f'{feat} → {cause}')
        axes[row, j].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Results Summary

In [ ]:
print('=' * 70)
print('SADHWANI ET AL. (2021) — RESULTS SUMMARY')
print('=' * 70)
print(f'\nArchitecture: {HIDDEN_SIZES}')
print(f'Ensemble size: {N_ENSEMBLE}')
print(f'Device: {device}')
print(f'\nCross-entropy (negative avg log-likelihood):')
train_probs = ensemble.predict_proba(X_train[:100000], device=device)
train_ce_approx = -np.mean(np.log(train_probs[np.arange(len(train_probs)), y_train[:100000]].clip(1e-10)))
print(f'  Train (approx): {train_ce_approx:.6f}')
print(f'  Val:            {val_ce:.6f}')
print(f'  Test:           {test_ce:.6f}')

print(f'\nAR models fitted: {list(ar_models.keys())}')

print(f'\n--- C-index (Frozen features — no future info) ---')
frozen_ci = cindex_df[cindex_df['Method'] == 'Frozen'].set_index('Cause')
print(frozen_ci[[f'C({t})' for t in EVAL_TIMES]].to_string())

print(f'\n--- C-index (AR-simulated macro — no future info) ---')
ar_ci = cindex_df[cindex_df['Method'] == 'AR-simulated'].set_index('Cause')
print(ar_ci[[f'C({t})' for t in EVAL_TIMES]].to_string())

print(f'\n--- C-index (Observed features — diagnostic) ---')
obs_ci = cindex_df[cindex_df['Method'] == 'Observed'].set_index('Cause')
print(obs_ci[[f'C({t})' for t in EVAL_TIMES]].to_string())

print(f'\n--- Brier Score (AR-simulated macro) ---')
ar_bs = brier_df[brier_df['Method'] == 'AR-simulated'].set_index('Cause')
print(ar_bs[[f'BS({t})' for t in EVAL_TIMES]].to_string())